# 03 - Multi-Trial Benchmark Evaluations (Champions & Classical Baselines)

This notebook consolidates the **Post-Evolution Benchmark Evaluation Pipeline** into a unified, reproducible workflow:
1. **Champion Extraction & Balance Check**: Selects top-performing candidate algorithms per problem condition from SQLite (`data/db.sqlite3`) and exports `data/champions.json`.
2. **Pre-Flight Diagnostic Audit**: Scans existing IOH traces in `results/evaluations/traces/` and reports workload status.
3. **LLaMEA Champions Benchmark Execution**: Executes $N=10$ independent evaluations per champion with IOHprofiler instrumentation.
4. **Classical Baselines Benchmark Execution**: Executes $N=10$ independent evaluations for baseline optimizers (`CMA-ES`, `DE`, `PSO`).

In [5]:
# Setup paths and services
# Ensure project root src/ is in sys.path
import os
import sys
from pathlib import Path

cwd = Path('.').resolve()
root_dir = cwd.parent if cwd.name == 'notebooks' else cwd
src_dir = root_dir / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

%load_ext autoreload
%autoreload 2

import json
import pandas as pd
from IPython.display import display, HTML

from shared.config import DATA_DIR, RESULTS_DIR
from benchmarking.application.selection_service import ChampionSelectionService
from benchmarking.application.evaluation_service import BenchmarkEvaluationService
from benchmarking.domain.baselines import BASELINES

champ_service = ChampionSelectionService()
eval_service = BenchmarkEvaluationService()
CHAMPIONS_PATH = DATA_DIR / 'champions.json'

# ── Optional Filters (Set to None to evaluate all completed tasks) ─────────
FILTER_MODELS     = None   # e.g., ['14B'], ['7B'], or None for all
FILTER_PROBLEMS   = None   # e.g., [1, 8], [8, 11, 15], or None for all
FILTER_STRATEGIES = None   # e.g., ['baseline', 'guided'], or None for all
FILTER_DIMS       = None   # e.g., [2, 3], [5], or None for all
FILTER_NOISE      = None   # e.g., [0.0, 0.05], or None for all
FILTER_BASELINES  = None   # e.g., ['CMA-ES', 'DE', 'PSO'], or None for all

print('✅ Benchmark evaluation environment initialized.')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Benchmark evaluation environment initialized.


## 1. Champion Selection & Experiment Balance Check
Query the synthesis database, inspect run balance, and export problem-specific champions to `data/champions.json`.

In [6]:
summary, total_completed = champ_service.get_experiment_balance()

if summary.empty:
    raise RuntimeError('No completed experiments found in database. Please run Notebook 02 first.')

print(f'== Total Completed Experiments: {total_completed} ==')
print(summary.to_string(index=False))

# Export champions
champions_dict, df_champions = champ_service.export_champions(CHAMPIONS_PATH)
print(f'\n✅ Exported {len(df_champions)} champions to {CHAMPIONS_PATH}')


== Total Completed Experiments: 292 ==
 problem_id  dim  noise_std prompt_strategy  count
          1    2       0.00        baseline      3
          1    2       0.00          guided      2
          1    2       0.00        thinking      2
          1    2       0.00   vectorization      2
          1    2       0.05        baseline      2
          1    2       0.05          guided      5
          1    2       0.05        thinking      2
          1    2       0.05   vectorization      2
          1    3       0.00        baseline      3
          1    3       0.00          guided      5
          1    3       0.00        thinking      2
          1    3       0.00   vectorization      2
          1    3       0.05        baseline      3
          1    3       0.05          guided      3
          1    3       0.05        thinking      2
          1    3       0.05   vectorization      2
          1    5       0.00        baseline      3
          1    5       0.00          guided

## 2. Pre-Flight Diagnostic Audit (Champions & Baselines Workload)
Audit pending vs. completed evaluation runs across all experimental conditions.

In [7]:
# 1. Load champions
with open(CHAMPIONS_PATH, 'r', encoding='utf-8') as f:
    champions_raw = json.load(f)
champions_flat = eval_service.champions_repo.get_champions_flat(champions_raw)

# ── Embedded HTML Dashboard Helper (Notebook-Local UI) ─────────────────────────
def render_html_dashboard(
    df_audit: pd.DataFrame,
    title: str = "Benchmark Evaluation Pre-Flight Audit",
    subtitle: str = "Real-time status of empirical evaluation runs",
    group_column: str = "model",
) -> str:
    if df_audit.empty:
        return "<div>No audit data available.</div>"

    grp_col = group_column if group_column in df_audit.columns else ("baseline" if "baseline" in df_audit.columns else df_audit.columns[0])
    summary_rows = []
    for name, grp in df_audit.groupby(grp_col):
        total = len(grp)
        completed = len(grp[grp["status"] == "COMPLETED"])
        pending = len(grp[grp["status"] == "PENDING"])
        needs_rerun = len(grp[grp["status"] == "NEEDS_RERUN"])
        missing_code = len(grp[grp["status"] == "MISSING_CODE"])
        to_run_mask = grp["status"].isin(["PENDING", "NEEDS_RERUN"])
        if "is_filtered" in grp.columns:
            to_run_mask = to_run_mask & (~grp["is_filtered"])
        to_run = len(grp[to_run_mask])
        pct = (completed / total * 100) if total > 0 else 0.0
        summary_rows.append({
            "Group": str(name).upper(),
            "Total Tasks": total,
            "Completed": completed,
            "To Run": to_run,
            "Needs Rerun": needs_rerun,
            "Missing Code": missing_code,
            "Progress": pct,
        })

    tot_tasks = len(df_audit)
    tot_comp = len(df_audit[df_audit["status"] == "COMPLETED"])
    tot_to_run = len(df_audit[df_audit["status"].isin(["PENDING", "NEEDS_RERUN"])])
    overall_pct = (tot_comp / tot_tasks * 100) if tot_tasks > 0 else 0.0

    cards_html = f"""
    <div style="display: flex; gap: 15px; margin-bottom: 20px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <div style="flex: 1; background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px; padding: 15px; text-align: center;">
            <div style="font-size: 12px; color: #64748b; text-transform: uppercase; font-weight: 600;">Total Tasks</div>
            <div style="font-size: 24px; font-weight: 700; color: #0f172a; margin-top: 5px;">{tot_tasks}</div>
        </div>
        <div style="flex: 1; background: #f0fdf4; border: 1px solid #bbf7d0; border-radius: 8px; padding: 15px; text-align: center;">
            <div style="font-size: 12px; color: #166534; text-transform: uppercase; font-weight: 600;">Completed</div>
            <div style="font-size: 24px; font-weight: 700; color: #15803d; margin-top: 5px;">{tot_comp}</div>
        </div>
        <div style="flex: 1; background: #fff7ed; border: 1px solid #fed7aa; border-radius: 8px; padding: 15px; text-align: center;">
            <div style="font-size: 12px; color: #9a3412; text-transform: uppercase; font-weight: 600;">To Execute</div>
            <div style="font-size: 24px; font-weight: 700; color: #c2410c; margin-top: 5px;">{tot_to_run}</div>
        </div>
        <div style="flex: 1; background: #f0f9ff; border: 1px solid #bae6fd; border-radius: 8px; padding: 15px; text-align: center;">
            <div style="font-size: 12px; color: #075985; text-transform: uppercase; font-weight: 600;">Overall Progress</div>
            <div style="font-size: 24px; font-weight: 700; color: #0284c7; margin-top: 5px;">{overall_pct:.1f}%</div>
        </div>
    </div>
    """

    rows_html = ""
    for r in summary_rows:
        bar_color = "#22c55e" if r["Progress"] >= 100 else ("#3b82f6" if r["Progress"] > 0 else "#94a3b8")
        rows_html += f"""
        <tr style="border-bottom: 1px solid #f1f5f9;">
            <td style="padding: 10px 14px; font-weight: 600; color: #1e293b;">{r['Group']}</td>
            <td style="padding: 10px 14px; text-align: center; color: #475569;">{r['Total Tasks']}</td>
            <td style="padding: 10px 14px; text-align: center; color: #15803d; font-weight: 600;">{r['Completed']}</td>
            <td style="padding: 10px 14px; text-align: center; color: #c2410c; font-weight: 600;">{r['To Run']}</td>
            <td style="padding: 10px 14px; text-align: center; color: #64748b;">{r['Needs Rerun']}</td>
            <td style="padding: 10px 14px;">
                <div style="display: flex; align-items: center; gap: 10px;">
                    <div style="flex: 1; background: #e2e8f0; border-radius: 9999px; height: 8px; overflow: hidden;">
                        <div style="background: {bar_color}; height: 100%; width: {r['Progress']}%;"></div>
                    </div>
                    <span style="font-size: 12px; font-weight: 600; color: #475569; width: 40px;">{r['Progress']:.0f}%</span>
                </div>
            </td>
        </tr>
        """

    table_html = f"""
    <table style="width: 100%; border-collapse: collapse; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 13px; background: white; border: 1px solid #e2e8f0; border-radius: 8px; overflow: hidden;">
        <thead>
            <tr style="background: #f8fafc; border-bottom: 2px solid #e2e8f0; color: #475569; font-weight: 600; text-align: left;">
                <th style="padding: 12px 14px;">Group / Model</th>
                <th style="padding: 12px 14px; text-align: center;">Total</th>
                <th style="padding: 12px 14px; text-align: center;">Completed</th>
                <th style="padding: 12px 14px; text-align: center;">To Run</th>
                <th style="padding: 12px 14px; text-align: center;">Needs Rerun</th>
                <th style="padding: 12px 14px;">Progress</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    """

    return f"""
    <div style="margin: 20px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
        <h3 style="margin: 0 0 4px 0; color: #0f172a; font-size: 18px; font-weight: 700;">{title}</h3>
        <p style="margin: 0 0 16px 0; color: #64748b; font-size: 13px;">{subtitle}</p>
        {cards_html}
        {table_html}
    </div>
    """

# 2. Audit champions workload
df_champ_audit = eval_service.audit_champions_workload(
    champions_flat=champions_flat,
    filter_models=FILTER_MODELS,
    filter_problems=FILTER_PROBLEMS,
    filter_strategies=FILTER_STRATEGIES,
    filter_dims=FILTER_DIMS,
    filter_noise=FILTER_NOISE,
)
print(f'🏆 Champions Workload: {len(df_champ_audit)} conditions ({df_champ_audit["status"].value_counts().to_dict()})')
display(HTML(render_html_dashboard(df_champ_audit, title="🏆 LLaMEA Champions Evaluation Audit", subtitle="Empirical benchmark evaluation status across LLM-evolved algorithms")))

# 3. Discover unique conditions for classical baselines
UNIQUE_CONFIGS = sorted(list({(c['dim'], c['noise_std'], c['problem_id']) for c in champions_flat.values()}))
df_base_audit = eval_service.audit_baselines_workload(
    conditions=UNIQUE_CONFIGS,
    baselines=FILTER_BASELINES,
    filter_dims=FILTER_DIMS,
    filter_noise=FILTER_NOISE,
    filter_problems=FILTER_PROBLEMS,
)
print(f'⚙️ Baselines Workload: {len(df_base_audit)} conditions ({df_base_audit["status"].value_counts().to_dict()})')
display(HTML(render_html_dashboard(df_base_audit, title="⚙️ Classical Baselines Evaluation Audit", subtitle="Empirical benchmark evaluation status across CMA-ES, DE, and PSO", group_column="baseline")))


🏆 Champions Workload: 227 conditions ({'COMPLETED': 227})


Group / Model,Total,Completed,To Run,Needs Rerun,Progress
QWEN_14B,108,108,0,0,100%
QWEN_7B,119,119,0,0,100%


⚙️ Baselines Workload: 90 conditions ({'COMPLETED': 90})


Group / Model,Total,Completed,To Run,Needs Rerun,Progress
CMAES,30,30,0,0,100%
DE,30,30,0,0,100%
PSO,30,30,0,0,100%


## 3. Execute LLaMEA Champions Benchmark (N=10 Independent Runs)
Evaluate evolved algorithms across target BBOB functions with full IOHprofiler `.dat` and `.json` logging.

In [8]:
results_champ_df = eval_service.run_champions(
    champions_flat=champions_flat,
    filter_models=FILTER_MODELS,
    filter_problems=FILTER_PROBLEMS,
    filter_strategies=FILTER_STRATEGIES,
    filter_dims=FILTER_DIMS,
    filter_noise=FILTER_NOISE,
    n_runs=10,
)
print(f'✅ LLaMEA Champions evaluation complete: {len(results_champ_df)} conditions processed.')


✅ LLaMEA Champions evaluation complete: 227 conditions processed.


## 4. Execute Classical Baselines Benchmark (CMA-ES, DE, PSO)
Run classical baseline optimizers across identical problem conditions for rigorous comparative benchmarking.

In [9]:
results_base_df = eval_service.run_baselines(
    conditions=UNIQUE_CONFIGS,
    baselines=FILTER_BASELINES,
    filter_dims=FILTER_DIMS,
    filter_noise=FILTER_NOISE,
    filter_problems=FILTER_PROBLEMS,
    n_runs=10,
)
print(f'✅ Classical Baselines evaluation complete: {len(results_base_df)} conditions processed.')


✅ Classical Baselines evaluation complete: 90 conditions processed.
